# Country Level Analysis

This notebook examines the raw JKP Global Factor Data emerging markets panel to understand the country
level composition of the cross section, the monthly firm counts per country, and the distribution of
missing data across countries. Column metadata is read from `data/processed/column_metadata.json`. The fields `retained_k0` and
  `retained_k1` give the characteristics that survived the training-set column filter.

In [ ]:
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

raw_path = Path("../data/Global Factor_EM.parquet")
col_metadata_path = Path("../data/processed/column_metadata.json")

train_end = "2015-12-31"
val_end = "2020-12-31"
missing_col_threshold = 0.30
missing_row_threshold = 1.0 / 3.0
min_firms_country = 20

raw_col_set = set(pq.read_schema(raw_path).names) if raw_path.exists() else set()

print(f"Column missing threshold (training set): {missing_col_threshold:.0%}")
print(f"Row missing threshold (per split): {missing_row_threshold:.1%}")

In [ ]:
id_cols = ["id", "gvkey", "iid", "eom", "excntry"]
df_ids = pd.read_parquet(raw_path, columns = id_cols)

print(f"Total observations: {len(df_ids):,}")
print(f"Date range: {df_ids['eom'].min()} to {df_ids['eom'].max()}")
print(f"Unique countries: {df_ids['excntry'].nunique()}")
print(f"Unique firms (by id): {df_ids['id'].nunique():,}")
print("Countries present:")
print(sorted(df_ids["excntry"].unique()))

#### Unique Firms per Country

In [ ]:
firms_per_country = (
    df_ids.groupby("excntry")["id"]
    .nunique()
    .sort_values(ascending = False)
    .reset_index()
    .rename(columns = {"id": "unique_firms"})
)

total_firms = firms_per_country["unique_firms"].sum()
firms_per_country["pct_of_total"] = (
    firms_per_country["unique_firms"] / total_firms * 100 ).round(2)
firms_per_country["cumulative_pct"] = firms_per_country["pct_of_total"].cumsum().round(2)

print(f"{'Country':<10} {'Unique Firms':>14} {'% of Total':>12} {'Cumulative %':>14}")
print()
for _, row in firms_per_country.iterrows():
    print(f"{row['excntry']:<10} {row['unique_firms']:>14,}{row['pct_of_total']:>11.2f}% {row['cumulative_pct']:>13.2f}%"
    )

print(f"{'Total':<10} {total_firms:>14,}")

### Monthly Firm Counts per Country

For each country we compute the maximum, mean, median, and minimum number of firms observed in any
single month. The Dual Path Transformer applies sparse cross-sectional attention within each country
independently, so the per-country peak firm count is the computationally binding quantity at each
rebalancing date.

In [ ]:
monthly_counts = (
    df_ids.groupby(["excntry", "eom"])["id"].nunique()
    .reset_index().rename(columns = {"id": "n_firms"})
)

country_monthly_stats = (
    monthly_counts.groupby("excntry")["n_firms"]
    .agg(["max", "mean", "median", "min"])
    .sort_values("max", ascending = False).reset_index()
)
country_monthly_stats.columns = [
    "excntry", "max_firms_month", "mean_firms_month",
    "median_firms_month", "min_firms_month"
]
country_monthly_stats["mean_firms_month"] = (
    country_monthly_stats["mean_firms_month"].round(0).astype(int)
)
country_monthly_stats["median_firms_month"] = (
    country_monthly_stats["median_firms_month"].round(0).astype(int)
)

print(f"{'Country':<10} {'Max':>8} {'Mean':>8} {'Median':>8} {'Min':>8}")
print()
for _, row in country_monthly_stats.iterrows():
    print(
        f"{row['excntry']:<10} {row['max_firms_month']:>8,}"
        f" {row['mean_firms_month']:>8,}{row['median_firms_month']:>8,}"
        f" {row['min_firms_month']:>8,}"
    )

### Cross Section Size Over Time

We aggregate firm counts across all countries for each calendar month and overlay the train and
validation split boundaries from `Config`. The lower subplot shows the peak within-country firm
count per month, which bounds the per-country attention computation.

In [ ]:
total_per_month = (
    monthly_counts.groupby("eom")["n_firms"]
    .sum().reset_index()
    .rename(columns = {"n_firms": "total_firms"})
)
max_country_per_month = (
    monthly_counts.groupby("eom")["n_firms"].max()
    .reset_index().rename(columns = {"n_firms": "max_country_firms"})
)

print(f"Peak global cross section: {total_per_month['total_firms'].max():,} firms")
print(f"Mean global cross section: {total_per_month['total_firms'].mean():,.0f} firms")
print(f"Median global cross section: {total_per_month['total_firms'].median():,.0f} firms")
print(f"Month of global peak: {total_per_month.loc[total_per_month['total_firms'].idxmax(), 'eom']}")

print(f"Peak within-country cross section: {max_country_per_month['max_country_firms'].max():,} firms")
print(f"Mean within-country peak: {max_country_per_month['max_country_firms'].mean():,.0f} firms")

t_train = pd.Timestamp(train_end)
t_val = pd.Timestamp(val_end)

fig, axes = plt.subplots(2, 1, figsize = (13, 8), sharex = True)

eom_dt = pd.to_datetime(total_per_month["eom"])
axes[0].plot(eom_dt, total_per_month["total_firms"], linewidth = 1.2)
axes[0].axvline(t_train, color = "green", linestyle = "--", linewidth = 1,
                label = f"train end ({train_end})")
axes[0].axvline(t_val, color = "blue", linestyle = "--", linewidth = 1,
                label = f"val end ({val_end})")
axes[0].set_ylabel("Total Firms in Cross Section")
axes[0].set_title("Global Cross Section Size per Month")
axes[0].legend(fontsize = 9)
axes[0].grid(alpha = 0.3)

eom_dt2 = pd.to_datetime(max_country_per_month["eom"])
axes[1].plot(eom_dt2, max_country_per_month["max_country_firms"],
             linewidth = 1.2, color = "C1")
axes[1].axvline(t_train, color = "green", linestyle = "--", linewidth = 1)
axes[1].axvline(t_val, color = "blue", linestyle = "--", linewidth = 1)
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Max Firms in Any Single Country")
axes[1].set_title("Peak Within-Country Cross Section Size per Month")
axes[1].grid(alpha = 0.3)

plt.tight_layout()
plt.show()

### Characteristic Column Classification

The pipeline defines
`metadata_cols` as the set of panel identifier, price, volume, and accounting aggregate columns
that are excluded from all characteristic processing steps. Every remaining raw column that is not
a target column is a candidate characteristic. Candidates listed in `k0_characteristic_list` are
market-based (K0); all others are accounting-based (K1).

The column filter in `filter_columns_by_missing` then drops any characteristic whose null rate in
the training set exceeds `missing_col_threshold = 0.30`. The retained column lists are written to
`column_metadata.json` and loaded here as the authoritative source. When that file is absent, we
fall back to the full candidate list (no filter applied).

In [ ]:
metadata_cols = [
    "permno", "permco", "gvkey", "iid", "id", "date", "excntry", "eom",
    "obs_main", "exch_main", "common", "primary_sec", "source_crsp",
    "size_grp", "me", "me_company", "prc", "prc_local", "prc_high",
    "prc_low", "bidask", "curcd", "fx", "gics", "naics", "sic", "ff49",
    "dolvol", "shares", "tvol", "adjfct", "comp_tpci", "crsp_shrcd",
    "comp_exchg", "crsp_exchcd", "ret", "ret_exc", "ret_local",
    "ret_exc_lead1m", "ret_lag_dif", "enterprise_value", "book_equity",
    "assets", "sales", "net_income", "intrinsic_value",
]

k0_characteristic_list = [
    "market_equity",
    "div1m_me", "div3m_me", "div6m_me", "div12m_me",
    "divspc1m_me", "divspc12m_me",
    "chcsho_1m", "chcsho_3m", "chcsho_6m", "chcsho_12m",
    "eqnpo_1m", "eqnpo_3m", "eqnpo_6m", "eqnpo_12m",
    "ret_1_0", "ret_3_1", "ret_6_1", "ret_9_1", "ret_12_1",
    "ret_12_7", "ret_60_12", "ret_2_0", "ret_3_0", "ret_6_0",
    "ret_9_0", "ret_12_0", "ret_18_1", "ret_24_1", "ret_24_12",
    "ret_36_1", "ret_36_12", "ret_48_1", "ret_48_12",
    "ret_60_1", "ret_60_36",
    "seas_1_1an", "seas_1_1na", "seas_2_5an", "seas_2_5na",
    "seas_6_10an", "seas_6_10na", "seas_11_15an", "seas_11_15na",
    "seas_16_20an", "seas_16_20na",
    "resff3_6_1", "resff3_12_1",
    "ivol_capm_21d", "ivol_capm_252d", "ivol_capm_60m",
    "ivol_ff3_21d", "ivol_hxz4_21d",
    "iskew_capm_21d", "iskew_ff3_21d", "iskew_hxz4_21d",
    "rvol_21d", "rvol_252d", "rvolhl_21d",
    "rmax1_21d", "rmax5_21d", "rmax5_rvol_21d",
    "rskew_21d", "coskew_21d",
    "beta_60m", "beta_21d", "beta_252d",
    "beta_dimson_21d", "betadown_252d", "betabab_1260d",
    "ami_126d", "dolvol_126d", "dolvol_var_126d",
    "turnover_126d", "turnover_var_126d",
    "zero_trades_21d", "zero_trades_126d", "zero_trades_252d",
    "bidaskhl_21d", "corr_1260d",
    "prc_highprc_252d",
    "age",
    "aliq_mat",
    "mispricing_mgmt", "mispricing_perf",
]

target_set = {"target_3m", "target_6m", "target_12m"}
metadata_set = set(metadata_cols)

candidate_chars_all = [
    c for c in sorted(raw_col_set)
    if c not in metadata_set and c not in target_set
]
k0_in_raw = [c for c in k0_characteristic_list if c in raw_col_set]
k1_in_raw = [c for c in candidate_chars_all if c not in set(k0_characteristic_list)]

print(f"Total raw columns: {len(raw_col_set)}")
print(f"Metadata columns found in raw: {sum(1 for c in metadata_cols if c in raw_col_set)}")
print(f"K0 candidates in raw data: {len(k0_in_raw)}")
print(f"K1 candidates in raw data: {len(k1_in_raw)}")
print(f"Total candidate characteristics: {len(candidate_chars_all)}")

In [ ]:
if col_metadata_path.exists():
    with open(col_metadata_path, "r") as f:
        col_meta = json.load(f)
    retained_k0 = col_meta["retained_k0"]
    retained_k1 = col_meta["retained_k1"]
    country_codes_meta = col_meta.get("country_codes", [])
    print(f"Column metadata loaded from: {col_metadata_path}")

    print(f"K0 candidates: {len(k0_in_raw)}, retained: {len(retained_k0)}, "
          f"dropped by {missing_col_threshold:.0%} filter: {len(k0_in_raw) - len(retained_k0)}")
    print(f"K1 candidates: {len(k1_in_raw)}, retained: {len(retained_k1)}, "
          f"dropped by {missing_col_threshold:.0%} filter: {len(k1_in_raw) - len(retained_k1)}")
    print(f"Countries in lookup: {len(country_codes_meta)}")
    chars_in_raw = [c for c in retained_k0 + retained_k1 if c in raw_col_set]
else:
    print(f"column_metadata.json not found at {col_metadata_path}.")
    print("Using all candidate characteristics (column filter not yet applied).")
    retained_k0 = k0_in_raw
    retained_k1 = k1_in_raw
    country_codes_meta = []
    chars_in_raw = candidate_chars_all

print()
print(f"Characteristic columns for downstream analysis: {len(chars_in_raw)}")

### Column-Level Missing Rate Analysis (Training Period)

We replicate the `filter_columns_by_missing` logic from the pipeline to confirm how many
characteristics in each group survive the training-set null rate threshold. Null rates are computed
on observations with `eom <= train_end`, consistent with the pipeline's look-ahead-free column
selection. The resulting distribution also serves as a diagnostic for whether the 30% threshold
is appropriate for the EM universe.

In [ ]:
train_read_cols = ["eom"] + candidate_chars_all
available_train_cols = [c for c in train_read_cols if c in raw_col_set]
print(f"Reading {len(available_train_cols) - 1} candidate columns for training-period analysis...")

df_train_chars = pd.read_parquet(
    raw_path,
    columns = available_train_cols,
    filters = [("eom", "<=", pd.Timestamp(train_end))],
)
print(f"Training period observations: {len(df_train_chars):,}")

actual_char_cols = [c for c in candidate_chars_all if c in df_train_chars.columns]
col_null_rates = (
    df_train_chars[actual_char_cols].isnull().mean()
    .sort_values(ascending = False)
    .rename("null_rate")
    .reset_index()
    .rename(columns = {"index": "column"})
)
col_null_rates["group"] = col_null_rates["column"].apply(
    lambda c: "K0" if c in set(k0_characteristic_list) else "K1"
)
col_null_rates["survives_filter"] = col_null_rates["null_rate"] <= missing_col_threshold

n_k0_survives = col_null_rates[col_null_rates["group"] == "K0"]["survives_filter"].sum()
n_k1_survives = col_null_rates[col_null_rates["group"] == "K1"]["survives_filter"].sum()
n_k0_total = (col_null_rates["group"] == "K0").sum()
n_k1_total = (col_null_rates["group"] == "K1").sum()


print(f"K0: {n_k0_survives} of {n_k0_total} survive the {missing_col_threshold:.0%} threshold")
print(f"K1: {n_k1_survives} of {n_k1_total} survive the {missing_col_threshold:.0%} threshold")


fig, ax = plt.subplots(figsize = (11, 4))
for group, color in [("K0", "C0"), ("K1", "C1")]:
    rates = col_null_rates[col_null_rates["group"] == group]["null_rate"]
    ax.hist(rates, bins = 30, alpha = 0.6, label = group, color = color)
ax.axvline(x = missing_col_threshold, color = "red", linestyle = "--", linewidth = 1.2,
           label = f"threshold = {missing_col_threshold:.0%}")
ax.set_xlabel("Null Rate in Training Period")
ax.set_ylabel("Number of Characteristics")
ax.set_title("Distribution of Training-Period Null Rates Across Characteristics")
ax.legend()
ax.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

del df_train_chars
gc.collect()

### Missing Data Analysis by Country

We load the retained characteristic columns from the raw panel to compute per-country missingness
rates. Using the retained columns rather than all candidates reflects the actual pipeline output:
`drop_high_missing_rows` applies the 1/3 row filter to the same retained column set that the model
will receive. Missingness rates computed here are therefore directly comparable to the row-level
filter that the pipeline applies per split.

In [ ]:
read_cols = ["id", "eom", "excntry"] + chars_in_raw
print(f"Retained characteristic columns for analysis: {len(chars_in_raw)}")

df_miss = pd.read_parquet(raw_path, columns = read_cols)
print(f"Loaded: {len(df_miss):,} rows")

#### Mean Missingness Rate per Country

In [ ]:
char_data = df_miss[chars_in_raw]
df_miss = df_miss.copy()
df_miss["miss_frac"] = char_data.isnull().sum(axis = 1) / len(chars_in_raw)

country_miss = (
    df_miss.groupby("excntry")["miss_frac"]
    .agg(["mean", "median", "std"])
    .sort_values("mean", ascending = False).reset_index()
)
country_miss.columns = ["excntry", "mean_miss_rate", "median_miss_rate", "std_miss_rate"]

print(f"{'Country':<10} {'Mean Miss %':>12} {'Median Miss %':>14} {'Std':>8}")
print()
for _, row in country_miss.iterrows():
    print(
        f"{row['excntry']:<10}{row['mean_miss_rate']*100:>11.2f}%"
        f"{row['median_miss_rate']*100:>13.2f}% {row['std_miss_rate']:>8.4f}"
    )

#### Proportion of Heavily Missing Observations per Country

We define a heavily missing observation as one in which more than one third of the retained
characteristics are null. This matches the `drop_high_missing_rows` threshold applied per split
in the pipeline (`threshold = 1/3`). The proportion reported here is computed on the raw data
before the split and row filter; it therefore represents an upper bound on the fraction that
would be dropped per country in each split.

In [ ]:
df_miss["heavily_missing"] = df_miss["miss_frac"] > missing_row_threshold

heavy_miss_by_country = (
    df_miss.groupby("excntry")
    .agg(
        total_obs = ("id", "count"),
        heavy_miss_obs = ("heavily_missing", "sum"),
    )
    .reset_index()
)
heavy_miss_by_country["heavy_miss_pct"] = (
    heavy_miss_by_country["heavy_miss_obs"] / heavy_miss_by_country["total_obs"] * 100
).round(2)
heavy_miss_by_country = heavy_miss_by_country.sort_values("heavy_miss_pct", ascending = False)

print(f"Row filter threshold: miss_frac > {missing_row_threshold:.4f} ({missing_row_threshold:.1%})")
print()
print(f"{'Country':<10} {'Total Obs':>12} {'Would Be Dropped':>16} {'Drop %':>10}")
print()
for _, row in heavy_miss_by_country.iterrows():
    print(
        f"{row['excntry']:<10} {int(row['total_obs']):>12,}"
        f" {int(row['heavy_miss_obs']):>16,} {row['heavy_miss_pct']:>9.2f}%"
    )

### Market Capitalisation Analysis

The country composite portfolio construction combines
independently formed country level portfolios using weights proportional to each country's
aggregate market equity that month. In `load_dataset`, the `me` column is loaded from the
processed parquet as a `market_cap` tensor and used by
`portfolio_simulation_country_composite`. We therefore examine the distribution of market equity
across countries to understand how composite weights are likely to be allocated.

In [ ]:
me_avail = "me" in raw_col_set

if me_avail:
    df_me = pd.read_parquet(raw_path, columns = ["id", "eom", "excntry", "me"])
    n_missing_me = int(df_me["me"].isnull().sum())
    df_me = df_me.dropna(subset = ["me"])
    print(f"Rows loaded: {len(df_me) + n_missing_me:,}")
    print(f"Rows with null me dropped: {n_missing_me:,}")
    print(f"Rows retained: {len(df_me):,}")

    country_me_monthly = (
        df_me.groupby(["excntry", "eom"])["me"].sum().reset_index()
        .rename(columns = {"me": "total_me"})
    )
    country_me_stats = (
        country_me_monthly.groupby("excntry")["total_me"]
        .agg(["mean", "median"])
        .sort_values("mean", ascending = False).reset_index()
    )
    country_me_stats.columns = ["excntry", "mean_me", "median_me"]

    total_mean_me = country_me_stats["mean_me"].sum()
    country_me_stats["me_share_pct"] = (
        country_me_stats["mean_me"] / total_mean_me * 100
    ).round(2)
    country_me_stats["me_cumulative_pct"] = (
        country_me_stats["me_share_pct"].cumsum().round(2)
    )

    print()
    print(f"{'Country':<10} {'Mean Agg ME':>14} {'ME Share %':>12} {'Cumulative %':>14}")
    print()
    for _, row in country_me_stats.iterrows():
        print(
            f"{row['excntry']:<10} {row['mean_me']:>14,.0f}"
            f" {row['me_share_pct']:>11.2f}% {row['me_cumulative_pct']:>13.2f}%"
        )
else:
    print("me column not present in raw data.")
    print(
        "load_dataset will print a WARNING and fall back to firm count weighting "
        "for the country composite simulation."
    )
    country_me_stats = pd.DataFrame(
        columns = ["excntry", "mean_me", "median_me", "me_share_pct"]
    )

`portfolio_simulation_country_composite` forms a within-country quintile portfolio only for
countries with at least `config.min_firms_country = 20` firms in the cross section that month.
Countries below this threshold in a given month are excluded from the composite, and the
remaining countries' market capitalisation weights are renormalised. We compute the fraction of panel months in which each country meets this threshold.

In [ ]:
total_months = monthly_counts["eom"].nunique()

eligible_months = (
    monthly_counts[monthly_counts["n_firms"] >= min_firms_country]
    .groupby("excntry")["eom"].count().reset_index()
    .rename(columns = {"eom": "n_eligible_months"})
)
eligible_months["pct_eligible"] = (
    eligible_months["n_eligible_months"] / total_months * 100
).round(1)
eligible_months = eligible_months.sort_values(
    "pct_eligible", ascending = False
).reset_index(drop = True)

print(f"min_firms_country = {min_firms_country}  |  Total panel months: {total_months}")
print()
print(f"{'Country':<10} {'Eligible Months':>16} {'% of Panel':>12}")
print()
for _, row in eligible_months.iterrows():
    print(
        f"{row['excntry']:<10} {row['n_eligible_months']:>16,} {row['pct_eligible']:>11.1f}%"
    )

always_eligible = (
    eligible_months[eligible_months["pct_eligible"] >= 95.0]["excntry"].tolist()
)
sometimes_ineligible = (
    eligible_months[eligible_months["pct_eligible"] < 95.0]["excntry"].tolist()
)
print()
print(
    f"Always eligible (>= 95% of months), {len(always_eligible)}: "
    f"{sorted(always_eligible)}"
)
print(
    f"Sometimes below threshold, {len(sometimes_ineligible)}: "
    f"{sorted(sometimes_ineligible)}"
)

### Summary and Country Exclusion Analysis

We merge all country level statistics into a single summary table and perform a progressive
exclusion analysis. Countries are dropped in order of decreasing peak monthly firm count, and
after each removal we track the global cross section peak and the aggregate missingness rate of
the retained universe.

In [ ]:
summary = firms_per_country.merge(country_monthly_stats, on = "excntry")
summary = summary.merge(country_miss[["excntry", "mean_miss_rate"]], on = "excntry")
summary = summary.merge(heavy_miss_by_country[["excntry", "heavy_miss_pct"]], on = "excntry")
summary = summary.merge(
    eligible_months[["excntry", "n_eligible_months", "pct_eligible"]],
    on = "excntry", how = "left"
)
if me_avail and len(country_me_stats) > 0:
    summary = summary.merge(
        country_me_stats[["excntry", "me_share_pct"]], on = "excntry", how = "left"
    )
summary = summary.sort_values("max_firms_month", ascending = False).reset_index(drop = True)

display_cols = [
    "excntry", "unique_firms", "max_firms_month",
    "mean_miss_rate", "heavy_miss_pct", "pct_eligible"
]
if "me_share_pct" in summary.columns:
    display_cols.append("me_share_pct")

print(summary[display_cols].to_string(index = False, float_format = lambda x: f"{x:.2f}"))

#### Progressive Exclusion by Peak Firm Count

Countries are dropped one at a time in order of decreasing peak monthly firm count.

In [ ]:
all_countries = summary["excntry"].tolist()

print("Progressive country exclusion, dropping largest peak contributors first")
print()
print(f"{'Countries Dropped':<40} {'Remaining':>10} {'Peak Firms':>12} {'Mean Miss %':>12}")

dropped = []
for i in range(len(all_countries) + 1):
    remaining_countries = [c for c in all_countries if c not in dropped]
    if not remaining_countries:
        break

    mask = monthly_counts["excntry"].isin(remaining_countries)
    peak = monthly_counts[mask].groupby("eom")["n_firms"].sum().max()
    remaining_miss = (
        df_miss[df_miss["excntry"].isin(remaining_countries)]["miss_frac"].mean()
    )

    drop_label = ", ".join(dropped) if dropped else "(none)"
    if len(drop_label) > 38:
        drop_label = drop_label[:35] + "..."

    print(
        f"{drop_label:<40} {len(remaining_countries):>9}"
        f" {peak:>12,} {remaining_miss*100:>11.2f}%"
    )

    if i < len(all_countries):
        dropped.append(all_countries[i])

#### Target-Based Exclusion

We identify the minimum set of countries to exclude to bring the global peak below a target.

In [ ]:
target_max_firms = 10000

candidates = summary.sort_values("max_firms_month", ascending = False)
dropped_for_target = []
remaining = set(all_countries)

current_peak = monthly_counts.groupby("eom")["n_firms"].sum().max()
print(f"Starting peak: {current_peak:,} firms across {len(remaining)} countries")
print(f"Target: {target_max_firms:,} firms")
print()

for _, row in candidates.iterrows():
    if current_peak <= target_max_firms:
        break
    country = row["excntry"]
    remaining.discard(country)
    dropped_for_target.append(country)

    mask = monthly_counts["excntry"].isin(remaining)
    current_peak = monthly_counts[mask].groupby("eom")["n_firms"].sum().max()

    print(
        f"Drop {country:<6} (max {int(row['max_firms_month']):>5,}/month, "
        f"miss rate {row['mean_miss_rate']*100:.1f}%) "
        f"=> peak now {current_peak:,} firms, {len(remaining)} countries remain"
    )

print()
if current_peak <= target_max_firms:
    print(f"Target achieved: peak {current_peak:,} firms with {len(remaining)} countries")
    print(f"Countries dropped ({len(dropped_for_target)}): {', '.join(dropped_for_target)}")
    print(f"Countries retained ({len(remaining)}): {', '.join(sorted(remaining))}")
else:
    print(f"Could not reach target. Current peak: {current_peak:,}")

### Summary Visualisations

A six-panel layout covers firm count, market cap share, data quality, and composite eligibility.
The market cap panel is suppressed when the `me` column is absent from the raw data.

In [ ]:
has_me = me_avail and len(country_me_stats) > 0

fig, axes = plt.subplots(2, 3, figsize = (21, 10))
ax_list = axes.flatten()

ax = ax_list[0]
plot_data = firms_per_country.sort_values("unique_firms", ascending = True)
ax.barh(plot_data["excntry"], plot_data["unique_firms"])
ax.set_xlabel("Unique Firms")
ax.set_title("Unique Firms per Country")
ax.grid(axis = "x", alpha = 0.3)

ax = ax_list[1]
plot_data = country_monthly_stats.sort_values("max_firms_month", ascending = True)
ax.barh(plot_data["excntry"], plot_data["max_firms_month"])
ax.set_xlabel("Max Firms in Any Month")
ax.set_title("Peak Monthly Firm Count per Country")
ax.grid(axis = "x", alpha = 0.3)

if has_me:
    ax = ax_list[2]
    plot_data = country_me_stats.sort_values("me_share_pct", ascending = True)
    ax.barh(plot_data["excntry"], plot_data["me_share_pct"])
    ax.set_xlabel("Time-Averaged Market Cap Share (%)")
    ax.set_title("Market Capitalisation Share per Country")
    ax.grid(axis = "x", alpha = 0.3)
else:
    ax_list[2].set_visible(False)

ax = ax_list[3]
plot_data = country_miss.sort_values("mean_miss_rate", ascending = True)
ax.barh(plot_data["excntry"], plot_data["mean_miss_rate"] * 100)
ax.set_xlabel("Mean Missing Rate (%)")
ax.set_title("Mean Characteristic Missingness per Country")
ax.grid(axis = "x", alpha = 0.3)

ax = ax_list[4]
plot_data = heavy_miss_by_country.sort_values("heavy_miss_pct", ascending = True)
ax.barh(plot_data["excntry"], plot_data["heavy_miss_pct"])
ax.set_xlabel(f"Obs with > {missing_row_threshold:.0%} Missing (%)")
ax.set_title("Proportion Exceeding Row Filter Threshold")
ax.grid(axis = "x", alpha = 0.3)

ax = ax_list[5]
plot_data = eligible_months.sort_values("pct_eligible", ascending = True)
ax.barh(plot_data["excntry"], plot_data["pct_eligible"])
ax.axvline(x = 95.0, color = "red", linestyle = ":", linewidth = 1, label = "95%")
ax.set_xlabel("Months Meeting min_firms_country Threshold (%)")
ax.set_title(f"Country Composite Eligibility (min_firms = {min_firms_country})")
ax.legend(fontsize = 8)
ax.grid(axis = "x", alpha = 0.3)

plt.tight_layout()
plt.show()

In [ ]:
del df_ids, df_miss, char_data
if me_avail:
    del df_me, country_me_monthly
gc.collect()
print("Done.")